# stochastic_path

> 对应代码：`EconometricsCode/code_in_notes/Chap.25/stochastic_path.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.25`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.25")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们清空内存并生成一个含 30 期的人工数据集，设定随机种子保证结果可复现，再用 _n 生成时间序号并声明为时间变量。

In [ ]:
%%stata
clear
set more off
set obs 30
set seed 505
gen t=_n
tsset t

接下来我们模拟 10 条随机游走（random walk）路径：每期先抽取标准正态冲击 rnormal()，再从第二期起把上一期的值累加上去，即 x_t = x_{t-1} + ε_t。随机游走是最简单的单位根过程，它的每条实现路径都不可预测地漂移，方差随时间发散，是理解非平稳性的基准例子。注意 forvalues 循环整体不可拆分。

In [ ]:
%%stata
forvalues i=1/10{
	gen x`i'=rnormal()
	replace x`i'=x`i'+L.x`i' if _n>1
}

最后把这 10 条路径画在一起：第一条路径用实线突出显示，并加上 t=10 附近的参考竖线，其余路径用点线；同时在 t=10 处用散点标出每条路径的取值。这张图直观展示了“随机游走的一次实现 vs 同一时点上的截面分布”：任一固定时点，各路径的取值构成该过程在该时点的截面分布，其离散程度随时间增大。

In [ ]:
%%stata
twoway (tsline x1, tline(9.8 10.2) ) (tsline x2-x10, lpattern(dot dot dot dot dot dot dot dot dot)) (scatter x1 t if t==10, msize(0.8) mc(red) ) (scatter x2 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow)) (scatter x3 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow))(scatter x4 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow))(scatter x5 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow))(scatter x6 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow))(scatter x7 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow))(scatter x8 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow))(scatter x9 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow))(scatter x10 t if t==10, msize(0.5) mc(purple) msymbol(circle_hollow)), legend(off)